# Replay benchmark: supplier-exception decisions

Run **Runtime → Run all** in Google Colab. This CPU-only notebook generates small synthetic cases and replays structured evidence updates through the actual pipeline. No API keys, customer data, or external dispatch are needed.

Compare constrained optimization with a deterministic greedy allocation baseline at each snapshot. Penalties are **synthetic weighted units**, not dollars or measured commercial savings. Repeated snapshots describe the same evolving operational world: never sum snapshot demand, shortage, or total cost as if each snapshot were an independent order book.

The first cell installs or upgrades from the repository only when the replay API is missing. A local checkout with its `src` on `PYTHONPATH` runs without network access.

In [ ]:
import importlib
import importlib.util
import subprocess
import sys

needs_install = importlib.util.find_spec("assumption_ops") is None
if not needs_install:
    import assumption_ops
    needs_install = not hasattr(assumption_ops, "ReplayCase")
if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "git+https://github.com/crypticsymmetry/improved-giggle.git",
    ])
    for module_name in list(sys.modules):
        if module_name == "assumption_ops" or module_name.startswith("assumption_ops."):
            del sys.modules[module_name]
    importlib.invalidate_caches()

import json
import platform
from pathlib import Path
from pprint import pprint
from time import perf_counter

import scipy
from assumption_ops.replay import run_replay
from assumption_ops.replay_fixtures import synthetic_cases
from assumption_ops.replay_io import (
    load_replay_case, save_replay_case, write_replay_report_csv,
)


## Generate reproducible cases

Twelve orders keep this demonstration small. The generator creates supplier-delay, stock-loss, demand-surge, and cost-shock regimes with a fixed seed. Inspect exported fixtures before treating a result as representative of your workload.

In [ ]:
cases = synthetic_cases(order_count=12, seed=1729)
assert len(cases) > 0
print(f"Generated {len(cases)} reproducible cases.")
pprint(cases[0])


## Replay the real pipeline

Each strategy is scored against the **same frozen initial incumbent** when computing disruption cost. This makes the objective comparison meaningful. Changes relative to the preceding optimized snapshot are reported separately as operational churn.

The deterministic greedy baseline is deliberately simple. These experiments establish correctness and illustrate tradeoffs; they do not establish superiority over commercial planning systems. Solver timing is machine-dependent and includes warm-up effects.

In [ ]:
environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "scipy": scipy.__version__,
    "order_count": 12,
    "seed": 1729,
    "external_dispatch": False,
}
started = perf_counter()
reports = [run_replay(case) for case in cases]
environment["total_replay_wall_seconds"] = perf_counter() - started
pprint(environment)


## Per-snapshot allocation quality

Weighted cost combines the configured acquisition, lateness, substitution, shortage, and disruption penalties. A lower objective can trade one service metric against another. Inspect on-time service and unfilled demand alongside cost.

`greedy_minus_optimized` is an objective difference in weighted units. If the optimizer proves optimality, it should not score worse than the feasible greedy baseline under the shared cost reference.

In [ ]:
snapshot_rows = []
for case_index, report in enumerate(reports):
    for step in report["steps"]:
        optimized = step["optimized"]
        greedy = step["greedy"]
        snapshot_rows.append({
            "case_index": case_index,
            "case_name": cases[case_index].name,
            "step": step["step"],
            "optimized_status": optimized["status"],
            "optimized_cost": optimized["evaluation"]["costs"]["total"],
            "greedy_cost": greedy["evaluation"]["costs"]["total"],
            "greedy_minus_optimized": step["greedy_minus_optimized"],
            "optimized_on_time": optimized["evaluation"]["metrics"]["on_time_units"],
            "greedy_on_time": greedy["evaluation"]["metrics"]["on_time_units"],
            "optimized_unfilled": optimized["evaluation"]["metrics"]["unfilled_units"],
            "greedy_unfilled": greedy["evaluation"]["metrics"]["unfilled_units"],
            "optimized_seconds": optimized["elapsed_seconds"],
            "greedy_seconds": greedy["elapsed_seconds"],
        })
        assert optimized["evaluation"]["metrics"]["requested_units"] == (
            greedy["evaluation"]["metrics"]["requested_units"]
        )
        if step["optimized_no_worse_when_proven"] is not None:
            assert step["optimized_no_worse_when_proven"]
pprint(snapshot_rows)


## Final snapshots: one order book per case

The final summaries below report the last snapshot only. They deliberately do not aggregate demand, shortage, service, or objective cost over repeated snapshots.

In [ ]:
final_rows = []
for case_index, report in enumerate(reports):
    summary = report["summary"]
    for strategy in ("optimized", "greedy"):
        evaluation = summary[f"final_{strategy}"]
        final_rows.append({
            "case_index": case_index,
            "case_name": cases[case_index].name,
            "strategy": strategy,
            "final_cost": evaluation["costs"]["total"],
            **evaluation["metrics"],
        })
pprint(final_rows)


## Review and disruption telemetry

Counts of confirmed fields, proposed decision lines, invalidated previous decisions, and changed allocation arcs indicate work that a reviewer may need to inspect. **These are workload proxies, not measured review time or labor savings.** Arc/unit churn can accumulate across transitions; it does not represent additional customer demand.

The replay records local decisions and audit history. It does not execute ERP actions or dispatch messages.

In [ ]:
review_rows = []
for case_index, report in enumerate(reports):
    for step in report["steps"]:
        review_rows.append({
            "case_index": case_index,
            "case_name": cases[case_index].name,
            "step": step["step"],
            "confirmed_fields": step["confirmed_fields"],
            "proposed_decision_lines": step["proposed_decision_lines"],
            "invalidated_previous_decisions": step["previous_decisions_invalidated"],
            "previous_active_decisions": step["previous_active_decision_count"],
            "changed_allocation_arcs": step["changed_allocation_arcs"],
            "changed_allocation_units": step["changed_allocation_units"],
            "duplicate_replay_verified": step["duplicate_replay_verified"],
        })
        if step["event_id"] is not None:
            assert step["duplicate_replay_verified"]
pprint(review_rows)


## Export reproducible fixtures and reports

Each JSON fixture can be loaded and replayed independently. JSON reports retain detailed allocations, accounting, and audit-related telemetry; CSV exposes per-snapshot strategy comparisons. Files are written into the notebook runtime's `replay_outputs` directory.

In [ ]:
output_dir = Path("replay_outputs")
output_dir.mkdir(exist_ok=True)
for case_index, (case, report) in enumerate(zip(cases, reports)):
    stem = f"case_{case_index:02d}"
    save_replay_case(case, output_dir / f"{stem}_fixture.json")
    (output_dir / f"{stem}_report.json").write_text(
        json.dumps(report, indent=2) + "\n", encoding="utf-8"
    )
    write_replay_report_csv(report, output_dir / f"{stem}_report.csv")
(output_dir / "environment.json").write_text(
    json.dumps(environment, indent=2) + "\n", encoding="utf-8"
)
print("Exported:")
for path in sorted(output_dir.iterdir()):
    print(path)


## Load your own case

First round-trip the exported fixture. To run your own case, replace `case_path` with the path to a JSON fixture using the same schema. Change supplies, orders, policies, or evidence batches in that file. No upload is required to execute this notebook as provided.

For actual operational data, assess input quality, policy calibration, feasible baseline strength, and human review outcomes separately. Synthetic results alone support none of those commercial conclusions.

In [ ]:
case_path = output_dir / "case_00_fixture.json"
loaded_case = load_replay_case(case_path)
loaded_report = run_replay(loaded_case)
assert loaded_report["summary"]["final_optimized"] == (
    reports[0]["summary"]["final_optimized"]
)
pprint(loaded_report["summary"])
